In [ ]:
import joblib
import pandas as pd
from src.model_utils import generate_shap_plots
import shap

# 1. Load best model and test data
model = joblib.load('../models/random_forest_model.pkl')
X_test = pd.read_csv('../data/processed/ecommerce_test_X.csv')
y_test = pd.read_csv('../data/processed/ecommerce_test_y.csv').values.ravel()

# 2. Get predictions to find TP, FP, FN
y_pred = model.predict(X_test)

results = pd.DataFrame({'actual': y_test, 'predicted': y_pred})

# Find indices for specific cases
tp_idx = results[(results.actual == 1) & (results.predicted == 1)].index[0] # Correctly caught fraud
fp_idx = results[(results.actual == 0) & (results.predicted == 1)].index[0] # False Alarm
fn_idx = results[(results.actual == 1) & (results.predicted == 0)].index[0] # Missed Fraud

print(f"Indices Found -> TP: {tp_idx}, FP: {fp_idx}, FN: {fn_idx}")

In [ ]:
#  Built-in vs. SHAP Importance
# 1. Built-in Importance
importances = pd.Series(model.feature_importances_, index=X_test.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(10,5))
importances.plot(kind='barh', color='skyblue').invert_yaxis()
plt.title("Built-in Feature Importance (Gini)")
plt.show()

# 2. SHAP Summary Plot (Global)
explainer, shap_vals, sample = generate_shap_plots(model, X_test, "E-commerce Fraud")

In [ ]:
# True Positive (Correctly identified Fraud)
# Force plot for a True Positive
shap.initjs()
shap.force_plot(explainer.expected_value[1], shap_vals[1][0], sample.iloc[0, :], matplotlib=True)
plt.show()